# Experiment 37 - ID Signal Attack

Goal: test whether the synthetic `id` column contains useful hidden structure.

37A is the exact Experiment 35A control. Then we add ID-derived features in small stages so we can see whether the ID actually helps.

The ID itself is unique, so exact-ID target encoding is not used. Instead, we test digit structure, modular groups, ranges, and leakage-safe group target/frequency encoding.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

TRAIN_PATH = '../data/train.csv'
TARGET = 'Will_Buy_EV'
ID_COL = 'id'

train = pd.read_csv(TRAIN_PATH)

target_values = train[TARGET].astype(str).str.strip()
y = target_values.map({'No': 0, 'Yes': 1}).astype(int)

numeric_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Environmental_Concern_Level'
]

categorical_cols = [
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level'
]

train_idx, valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

y_train = y.iloc[train_idx].reset_index(drop=True)
y_valid = y.iloc[valid_idx].reset_index(drop=True)

print('Train rows:', len(train_idx))
print('Validation rows:', len(valid_idx))


Train rows: 534932
Validation rows: 133733


In [2]:
def add_digit_identity_keys(df, source_cols):
    out = df.copy()

    for col in source_cols:
        values = pd.to_numeric(out[col], errors='coerce')
        safe = values.fillna(0).abs().astype(np.int64)
        strings = safe.astype(str)

        last2 = (safe % 100).astype(str)
        last3 = (safe % 1000).astype(str)
        digit_sum = strings.map(lambda x: sum(int(ch) for ch in x))
        first_last = strings.str[0] + '_' + strings.str[-1]
        count_last = strings.str.len().astype(str) + '_' + strings.str[-1]

        out[f'{col}__last2_key'] = last2
        out[f'{col}__last3_key'] = last3
        out[f'{col}__digit_sum_key'] = digit_sum.astype(str)
        out[f'{col}__first_last_key'] = first_last
        out[f'{col}__count_last_key'] = count_last

        missing = values.isna()
        for new_col in [
            f'{col}__last2_key',
            f'{col}__last3_key',
            f'{col}__digit_sum_key',
            f'{col}__first_last_key',
            f'{col}__count_last_key'
        ]:
            out.loc[missing, new_col] = '__MISSING__'

    return out


def add_id_features(df):
    out = df.copy()
    values = pd.to_numeric(out[ID_COL], errors='coerce')
    safe = values.fillna(0).abs().astype(np.int64)
    strings = safe.astype(str)

    out['id_last1'] = safe % 10
    out['id_last2'] = safe % 100
    out['id_last3'] = safe % 1000
    out['id_last4'] = safe % 10000
    out['id_first_digit'] = strings.str[0].astype(int)
    out['id_digit_count'] = strings.str.len()
    out['id_digit_sum'] = strings.map(lambda x: sum(int(ch) for ch in x))
    out['id_digit_sum_mod2'] = out['id_digit_sum'] % 2
    out['id_digit_sum_mod3'] = out['id_digit_sum'] % 3
    out['id_digit_sum_mod9'] = out['id_digit_sum'] % 9
    out['id_parity'] = safe % 2

    # Range groups at several granularities.
    out['id_bin_100'] = (safe // 100).astype(str)
    out['id_bin_1000'] = (safe // 1000).astype(str)
    out['id_bin_10000'] = (safe // 10000).astype(str)
    out['id_bin_50000'] = (safe // 50000).astype(str)

    out['id_last2_key'] = (safe % 100).astype(str)
    out['id_last3_key'] = (safe % 1000).astype(str)
    out['id_first_last_key'] = strings.str[0] + '_' + strings.str[-1]
    out['id_digit_sum_key'] = out['id_digit_sum'].astype(str)

    missing = values.isna()
    key_cols = [
        'id_last2_key', 'id_last3_key', 'id_first_last_key',
        'id_digit_sum_key', 'id_bin_100', 'id_bin_1000',
        'id_bin_10000', 'id_bin_50000'
    ]
    for col in key_cols:
        out.loc[missing, col] = '__MISSING__'

    return out


In [3]:
def smoothed_mapping(keys, target, smoothing=20):
    temp = pd.DataFrame({'key': keys.astype('string').fillna('__MISSING__'), 'target': target.values})
    stats = temp.groupby('key')['target'].agg(['count', 'mean'])
    global_mean = target.mean()
    return ((stats['count'] * stats['mean']) + (smoothing * global_mean)) / (stats['count'] + smoothing)


def add_oof_group_encoding(train_df, valid_df, y_train, key_cols, prefix):
    train_out = train_df.copy()
    valid_out = valid_df.copy()
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    for key_col in key_cols:
        te_col = f'{prefix}_{key_col}_te'
        freq_col = f'{prefix}_{key_col}_freq'
        train_out[te_col] = np.nan

        keys = train_df[key_col].astype('string').fillna('__MISSING__')

        for fold_train_pos, fold_valid_pos in skf.split(train_df, y_train):
            fold_keys = keys.iloc[fold_train_pos]
            fold_y = y_train.iloc[fold_train_pos]
            mapping = smoothed_mapping(fold_keys, fold_y, smoothing=20)
            valid_keys = keys.iloc[fold_valid_pos]
            train_out.loc[train_out.index[fold_valid_pos], te_col] = valid_keys.map(mapping).fillna(y_train.iloc[fold_train_pos].mean()).values

        full_mapping = smoothed_mapping(keys, y_train, smoothing=20)
        valid_keys = valid_df[key_col].astype('string').fillna('__MISSING__')
        valid_out[te_col] = valid_keys.map(full_mapping).fillna(y_train.mean()).values

        full_freq = keys.value_counts(dropna=False)
        train_out[freq_col] = keys.map(full_freq).fillna(0).values
        valid_out[freq_col] = valid_keys.map(full_freq).fillna(0).values

    return train_out, valid_out


In [4]:
def make_preprocessor(feature_df):
    num_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in feature_df.columns if c not in num_cols]

    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)
    ])


def run_xgb(X_train, X_valid, label):
    preprocessor = make_preprocessor(X_train)

    model = XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )

    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict_proba(X_valid)[:, 1]
    score = roc_auc_score(y_valid, pred)

    print(f'{label} ROC-AUC: {score:.6f}')
    return score


## 37A - Exact 35A control



In [5]:
base = train.drop(columns=[TARGET, ID_COL]).copy()
base_train = base.iloc[train_idx].reset_index(drop=True)
base_valid = base.iloc[valid_idx].reset_index(drop=True)

base_train = add_digit_identity_keys(base_train, [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km',
    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work'
])
base_valid = add_digit_identity_keys(base_valid, [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km',
    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work'
])

identity_keys = numeric_cols.copy()
base_train[identity_keys] = base_train[identity_keys].astype('string').fillna('__MISSING__')
base_valid[identity_keys] = base_valid[identity_keys].astype('string').fillna('__MISSING__')

base_train, base_valid = add_oof_group_encoding(
    base_train, base_valid, y_train, identity_keys, 'identity'
)

digit_keys = [c for c in base_train.columns if c.endswith('_key') and c != 'id']
base_train, base_valid = add_oof_group_encoding(
    base_train, base_valid, y_train, digit_keys, 'digit'
)

score_36a = run_xgb(base_train, base_valid, '36A_Control_35A')


36A_Control_35A ROC-AUC: 0.945084


## 37B - ID digit decomposition

Adds numeric structure from the ID without using the raw ID as a feature.


In [6]:
id_train_raw = train.iloc[train_idx].drop(columns=[TARGET]).reset_index(drop=True)
id_valid_raw = train.iloc[valid_idx].drop(columns=[TARGET]).reset_index(drop=True)

id_train = add_id_features(id_train_raw)
id_valid = add_id_features(id_valid_raw)

id_feature_cols = [c for c in id_train.columns if c.startswith('id_')]

X36b_train = base_train.copy()
X36b_valid = base_valid.copy()

for col in id_feature_cols:
    X36b_train[col] = id_train[col].values
    X36b_valid[col] = id_valid[col].values

score_36b = run_xgb(X36b_train, X36b_valid, '36B_ID_Digit_Decomposition')


36B_ID_Digit_Decomposition ROC-AUC: 0.944962


## 37C - Leakage-safe ID group encoding

Target/frequency encodings are added for ID ranges and digit groups. Training target encoding is OOF, while validation uses mappings learned only from the training split.


In [7]:
group_keys = [
    'id_last2_key',
    'id_last3_key',
    'id_first_last_key',
    'id_digit_sum_key',
    'id_bin_100',
    'id_bin_1000',
    'id_bin_10000',
    'id_bin_50000'
]

id_group_train = id_train[group_keys].copy()
id_group_valid = id_valid[group_keys].copy()

id_group_train, id_group_valid = add_oof_group_encoding(
    id_group_train, id_group_valid, y_train, group_keys, 'idgroup'
)

X36c_train = base_train.copy()
X36c_valid = base_valid.copy()

for col in id_group_train.columns:
    if col not in group_keys:
        X36c_train[col] = id_group_train[col].values
        X36c_valid[col] = id_group_valid[col].values

score_36c = run_xgb(X36c_train, X36c_valid, '36C_ID_Group_Encoding')


36C_ID_Group_Encoding ROC-AUC: 0.945016


## 37D - Combined ID attack

Combines the ID digit features and the leakage-safe ID group encodings.


In [8]:
X36d_train = X36b_train.copy()
X36d_valid = X36b_valid.copy()

for col in id_group_train.columns:
    if col not in group_keys:
        X36d_train[col] = id_group_train[col].values
        X36d_valid[col] = id_group_valid[col].values

score_36d = run_xgb(X36d_train, X36d_valid, '36D_Combined_ID_Attack')


36D_Combined_ID_Attack ROC-AUC: 0.945005


In [9]:
results = pd.DataFrame({
    'Experiment': [
        '36A_Control_35A',
        '36B_ID_Digit_Decomposition',
        '36C_ID_Group_Encoding',
        '36D_Combined_ID_Attack'
    ],
    'ROC_AUC': [score_36a, score_36b, score_36c, score_36d]
})

results['vs_23B'] = results['ROC_AUC'] - 0.945243
results['vs_35A'] = results['ROC_AUC'] - 0.945361

print(results.sort_values('ROC_AUC', ascending=False).to_string(index=False))

best = results.loc[results['ROC_AUC'].idxmax()]
print(f"\nBest: {best['Experiment']} -> {best['ROC_AUC']:.6f}")


                Experiment  ROC_AUC    vs_23B    vs_35A
           36A_Control_35A 0.945084 -0.000159 -0.000277
     36C_ID_Group_Encoding 0.945016 -0.000227 -0.000345
    36D_Combined_ID_Attack 0.945005 -0.000238 -0.000356
36B_ID_Digit_Decomposition 0.944962 -0.000281 -0.000399

Best: 36A_Control_35A -> 0.945084
